# Chapter 6: AI Agent for Autonomous Scraping

This notebook demonstrates the full autonomous scraping loop:
1. Agent receives a URL with no prior knowledge
2. Agent fetches and analyzes page structure
3. Agent generates a scraper config via LLM
4. Scraper executes the config
5. Agent validates results and suggests fixes

**Required:** Start demo sites + set `OPENROUTER_API_KEY` in `.env`

In [7]:
import sys
import json
import asyncio
import os
sys.path.insert(0, "../agents")

from dotenv import load_dotenv
load_dotenv("../.env")

# ── Backend selection ─────────────────────────────────────────────────────────
# Set to "ollama" to use your local/remote Ollama instance instead of OpenRouter
# Can also set LLM_BACKEND=ollama in .env
os.environ.setdefault("LLM_BACKEND", "openrouter")   # change to "ollama" to switch

from llm import chat, get_model, get_backend
print(f"Backend : {get_backend()}")
print(f"Model   : {get_model()}")

Backend : openrouter
Model   : mistralai/ministral-14b-2512


## 6.1 Step 1: Page Structure Analysis

In [8]:
import httpx
from bs4 import BeautifulSoup
from config_generator import summarise_html

TARGET_URL = "http://localhost:8001/products"  # ShopSphere SSR listing page

resp = httpx.get(TARGET_URL, headers={"User-Agent": "Mozilla/5.0"})
html = resp.text
soup = BeautifulSoup(html, "lxml")

visible_text = soup.get_text(strip=True)
is_csr = len(visible_text) < 200
print(f"URL: {TARGET_URL}")
print(f"Visible text length: {len(visible_text)} chars")
print(f"Detected as CSR: {is_csr}")
print(f"Render mode to use: {'playwright' if is_csr else 'static'}")
print()

summary = summarise_html(html, max_elements=30)
print("HTML structure summary (first 30 elements):")
print(summary)

URL: http://localhost:8001/products
Visible text length: 2222 chars
Detected as CSR: False
Render mode to use: static

HTML structure summary (first 30 elements):
<body> 'ShopSphereAll ProductsSSRProducts(50 items)CategoriesAll Pro'
  <header class="site-header"> 'ShopSphereAll ProductsSSR'
    <div class="container header-inner"> 'ShopSphereAll ProductsSSR'
      <a class="logo" href="/"> 'ShopSphere'
      <nav class="site-nav"> 'All Products'
        <a href="/"> 'All Products'
      <div class="header-badge"> 'SSR'
  <main class="container"> 'Products(50 items)CategoriesAll ProductsAccessoriesAudioBusi'
    <div class="page-header"> 'Products(50 items)'
      <h1 class="page-title"> 'Products(50 items)'
        <span class="page-count"> '(50 items)'
    <div class="layout-sidebar"> 'CategoriesAll ProductsAccessoriesAudioBusiness BooksCamerasD'
      <aside class="sidebar"> 'CategoriesAll ProductsAccessoriesAudioBusiness BooksCamerasD'
        <h3 class="sidebar-title"> 'Categories'

## 6.2 Step 2: LLM Config Generation

In [9]:
from config_generator import generate_config

print(f"Generating config for: {TARGET_URL}")
config = generate_config(TARGET_URL)

print(json.dumps(config, indent=2))

Generating config for: http://localhost:8001/products


Fetching listing page http://localhost:8001/products

Fetching sample detail page http://localhost:8001/products/macbook-pro-14

Detail page summary (45 lines)

Listing page summary (41 lines)

Asking LLM to generate config...

{
  "name": "ShopSphere_Products",
  "description": "Scraper for ShopSphere marketplace products with pagination support",
  "render_mode": "static",
  "sources": [
    {
      "url_template": "http://localhost:8001/products?page={n}",
      "index_type": "page",
      "pagination": {
        "start": 1,
        "step": 1,
        "max_pages": 20,
        "stop_condition": "no_results"
      }
    }
  ],
  "listing": {
    "link_selector": "article.product-card a.product-link",
    "link_prefix": "http://localhost:8001"
  },
  "fields": {
    "title": {
      "selector": "h1.product-title",
      "retrieve": "plaintext"
    },
    "price": {
      "selector": "span.price-amount",
      "retrieve": "plaintext"
    },
    "rating": {
      "selector": "span.rating-value",
      "retrieve": "plaintext"
    },
    "category": {
      "selector": "span.product-category-badge.large",
      "retrieve": "plaintext"
    },
    "in_stock": {
      "selector": "span.stock-status",
      "retrieve

## 6.3 Step 3: Execute the Config

In [10]:
from autonomous_scraper import run

print("Running scrape...")
results = await run(TARGET_URL, config, max_items=5)

print(f"\nScraped {len(results)} records")
for r in results[:3]:
    print(f"\n  {json.dumps({k: v for k, v in r.items() if k != '_errors'}, indent=4)[:300]}")

Output()

Running scrape...



Scraped 5 records

  {
    "title": "MacBook Pro 14\"",
    "price": "$1999.00",
    "rating": "4.8",
    "category": "Laptops",
    "in_stock": "In Stock",
    "image": "https://picsum.photos/seed/macbook-pro-14/600/380",
    "description": "The most powerful MacBook Pro ever. M3 Pro chip with 18GB unified memory deliv

  {
    "title": "Dell XPS 15",
    "price": "$1499.00",
    "rating": "4.6",
    "category": "Laptops",
    "in_stock": "In Stock",
    "image": "https://picsum.photos/seed/dell-xps-15/600/380",
    "description": "Premium Windows laptop with OLED display and Intel Core i7 processor. Ideal for creati

  {
    "title": "Lenovo ThinkPad X1 Carbon Gen 11",
    "price": "$1299.00",
    "rating": "4.7",
    "category": "Laptops",
    "in_stock": "In Stock",
    "image": "https://picsum.photos/seed/lenovo-thinkpad-x1-carbon/600/380",
    "description": "Business ultrabook with legendary ThinkPad durabili


## 6.4 Step 4: LLM Quality Validation

In [11]:
from autonomous_scraper import validate_results

if results:
    feedback = validate_results(results, config)
    print("LLM quality assessment:")
    print(feedback)

LLM quality assessment:
**Issues Identified:**

1. **Missing Critical Fields**: None (all records have all config fields).

2. **Malformed Values**:
   - **`ram` & `storage`**: All 3 records incorrectly repeat `processor` value (e.g., `"Apple M3 Pro"` instead of actual RAM/storage specs like `"18GB"` or `"512GB SSD"`).

3. **Extraction Errors**:
   - **`price`**: No numeric validation (e.g., `"$1999.00"` should be parsed as `1999.00`).
   - **`rating`**: No validation for numeric range (e.g., `4.8` should be `0.0–5.0`).
   - **`in_stock`**: String value (`"In Stock"`) should ideally be boolean (`true/false`).
   - **`tags`**: No validation for duplicates or consistency (e.g., `"laptop"` appears in all 3).


## 6.5 Autonomous Config Refinement Loop

In [12]:
async def autonomous_refine(
    url: str,
    max_retries: int = 2,
    max_items: int = 5,
) -> tuple[dict, list]:
    """
    Full autonomous scraping loop with LLM-driven refinement.
    1. Generate config
    2. Run scrape
    3. If results look bad, ask LLM to fix config
    4. Retry
    """
    config = generate_config(url)
    print(f"Generated config: {config.get('name')}")
    print(f"Backend: {get_backend()} / {get_model()}")

    for attempt in range(max_retries + 1):
        print(f"\nAttempt {attempt + 1}/{max_retries + 1}")
        results = await run(url, config, max_items=max_items)

        if not results:
            print("No results. Asking LLM to fix config...")
            fix_messages = [
                {"role": "system", "content": "Fix this scraper config. The scrape returned no results. Return only valid JSON config."},
                {"role": "user", "content": f"URL: {url}\nConfig: {json.dumps(config)}\nError: No items scraped. Check link_selector and pagination."}
            ]
            raw = chat(fix_messages, max_tokens=1024)
            raw = raw.strip().strip("```json").strip("```").strip()
            try:
                config = json.loads(raw)
                print("Config updated by LLM")
            except Exception:
                print("LLM fix invalid JSON, stopping")
                break
        else:
            non_null_fields = sum(
                1 for r in results
                for k, v in r.items()
                if k not in ("source_url", "_errors") and v is not None
            )
            total_fields = len(results) * len(config.get("fields", {}))
            quality = non_null_fields / total_fields if total_fields else 0
            print(f"Quality score: {quality:.1%} ({non_null_fields}/{total_fields} fields populated)")

            if quality > 0.5:
                print("Quality sufficient. Done.")
                return config, results

    return config, results


final_config, final_results = await autonomous_refine(
    "http://localhost:8003/jobs",  # JobHive SSR listing page
    max_retries=1,
    max_items=3,
)

print(f"\nFinal: {len(final_results)} records")
if final_results:
    print(json.dumps(final_results[0], indent=2))

Fetching listing page http://localhost:8003/jobs

Fetching sample detail page http://localhost:8003/jobs/senior-backend-engineer-stripe

Detail page summary (45 lines)

Listing page summary (47 lines)

Asking LLM to generate config...

Output()

Generated config: JobHiveJobsScraper
Backend: openrouter / mistralai/ministral-14b-2512

Attempt 1/2


Quality score: 91.7% (33/36 fields populated)
Quality sufficient. Done.

Final: 3 records
{
  "title": "Senior Backend Engineer",
  "company": "Stripe",
  "location": "San Francisco, CA",
  "salary": "$180,000 - $240,000",
  "job_type": "Full-time",
  "tags": null,
  "department": "Engineering",
  "experience": "5+ years",
  "description": "Build the financial infrastructure of the internet. You will own large sections of Stripe's payment processing pipeline, improving reliability, performance, and developer experience for millions of businesses worldwide.",
  "requirements": [
    "Python or Go",
    "Distributed systems",
    "PostgreSQL",
    "gRPC",
    "High-availability systems"
  ],
  "benefits": [
    "Equity",
    "Health + Dental + Vision",
    "401k matching",
    "Annual learning budget",
    "Home office stipend"
  ],
  "job_id": "JOB-001",
  "_errors": {
    "tags": "selector 'div.job-tags span.tag' matched nothing"
  },
  "source_url": "http://localhost:8003/jobs/senior-

## 6.6 CSR vs SSR: The Agent Handles Both

In [13]:
# The agent detects rendering mode and adjusts automatically
for site_name, site_url in [
    ("ShopSphere SSR", "http://localhost:8001"),
    ("ShopSphere CSR", "http://localhost:8002"),
]:
    print(f"\n{'='*50}")
    print(f"Analyzing: {site_name} at {site_url}")
    
    resp = httpx.get(site_url, headers={"User-Agent": "Mozilla/5.0"})
    text = BeautifulSoup(resp.text, "lxml").get_text(strip=True)
    render_mode = "playwright" if len(text) < 200 else "static"
    
    print(f"  Detected render mode: {render_mode}")
    print(f"  Visible text chars: {len(text)}")
    if render_mode == "playwright":
        print("  -> Agent will use headless browser for this site")
    else:
        print("  -> Agent will use fast static HTTP fetch for this site")


Analyzing: ShopSphere SSR at http://localhost:8001
  Detected render mode: static
  Visible text chars: 2222
  -> Agent will use fast static HTTP fetch for this site

Analyzing: ShopSphere CSR at http://localhost:8002
  Detected render mode: playwright
  Visible text chars: 159
  -> Agent will use headless browser for this site


## 6.7 Key Takeaways

1. **The full autonomous loop**: URL in, structured data out, zero manual config writing
2. **LLMs can analyze HTML structure** and produce valid selector-based configs
3. **Auto-detection of render mode** means the agent adapts to SSR and CSR sites alike
4. **Refinement loops** let the agent fix its own mistakes using error feedback
5. **The JSON config is the bridge**: humans can inspect and override what the AI generated
6. **This is the foundation of autonomous scraping agents** that can be given any URL and return structured data